In [ ]:
# Phase 2 laterality census: header-only pass over every study/series in the
# real training set. Answers the Phase 2 gate's two open numbers before any
# prep.py storage decisions get made: (1) laterality resolution coverage --
# what fraction of studies resolve via ImageLaterality / Laterality /
# string-match, what fraction conflict across series, what fraction stay
# unknown -- and (2) the real slice-count-per-series distribution the K=24
# slice budget should be sized against. CPU-only, internet off: this is a
# metadata read, not a training run, so it doesn't touch the GPU quota and
# doesn't need the accelerator-lottery ritual Phase 1 training needed.
import glob, os, shutil, sys, time

GIT_SHA = 'REPLACE_AT_PUSH_TIME'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import census_study_laterality
print('knee package imported successfully from', PKG)


In [ ]:
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_uids = set(train_df.loc[train_df[gold_cols].notna().any(axis=1), 'StudyInstanceUID'])
print(f'{len(train_df)} total studies, {len(gold_uids)} gold-labeled')

TRAIN_SERIES_DIR = f'{COMP_DIR}/train_series'
study_dirs = sorted(
    os.path.join(TRAIN_SERIES_DIR, d)
    for d in os.listdir(TRAIN_SERIES_DIR)
    if os.path.isdir(os.path.join(TRAIN_SERIES_DIR, d))
)
print(f'{len(study_dirs)} study directories found on disk')


In [ ]:
from pathlib import Path

rows = []
series_route_rows = []
t0 = time.time()

for i, study_dir in enumerate(study_dirs):
    study_uid = os.path.basename(study_dir)
    result = census_study_laterality(Path(study_dir))
    rows.append({
        'StudyInstanceUID': study_uid,
        'is_gold': study_uid in gold_uids,
        'n_series': result['n_series'],
        'side': result['side'],
        'route': result['route'],
        'slice_counts': result['slice_counts'],
    })
    for side, route in result['series_resolutions']:
        series_route_rows.append({'StudyInstanceUID': study_uid, 'side': side, 'route': route})
    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        print(f'{i + 1}/{len(study_dirs)} studies censused, {elapsed:.1f}s elapsed')

elapsed = time.time() - t0
print(f'done: {len(study_dirs)} studies in {elapsed:.1f}s ({elapsed / len(study_dirs) * 1000:.1f} ms/study)')

census_df = pd.DataFrame(rows)
series_route_df = pd.DataFrame(series_route_rows)


In [ ]:
# Study-level resolution coverage -- the Phase 2 gate's headline number.
print('Study-level route counts (all studies):')
print(census_df['route'].value_counts())
print()
print('Study-level route counts (58 gold studies only):')
print(census_df.loc[census_df['is_gold'], 'route'].value_counts())
print()
unknown_frac = (census_df['route'] == 'unknown').mean()
conflict_frac = (census_df['route'] == 'conflict').mean()
print(f'unknown fraction: {unknown_frac:.3%}')
print(f'conflict fraction: {conflict_frac:.3%}')


In [ ]:
# Route-vs-route agreement: among series where more than one route *could*
# resolve laterality (i.e. this series has ImageLaterality and we separately
# check what Laterality/string-match would have said), how often do they
# actually agree? resolve_laterality always returns the first route that
# fires, so this needs a direct per-route comparison, not just the winner.
import re
from knee.dicom import read_laterality_header

def _all_routes(header):
    out = {}
    if header.get('ImageLaterality'):
        out['ImageLaterality'] = header['ImageLaterality']
    if header.get('Laterality'):
        out['Laterality'] = header['Laterality']
    for field in ('SeriesDescription', 'BodyPartExamined'):
        text = header.get(field)
        if text and re.search(r'right', text, re.IGNORECASE):
            out[field] = 'R'
        elif text and re.search(r'left', text, re.IGNORECASE):
            out[field] = 'L'
    return out

agree, disagree = 0, 0
disagreement_examples = []
sample_n = min(3000, len(study_dirs))
for study_dir in study_dirs[:sample_n]:
    study_uid = os.path.basename(study_dir)
    for series_dir in sorted(Path(study_dir).iterdir()):
        if not series_dir.is_dir():
            continue
        dcm_files = sorted(series_dir.glob('*.dcm'))
        if not dcm_files:
            continue
        header = read_laterality_header(dcm_files[0])
        routes = _all_routes(header)
        if len(routes) < 2:
            continue
        values = set(routes.values())
        if len(values) == 1:
            agree += 1
        else:
            disagree += 1
            if len(disagreement_examples) < 10:
                disagreement_examples.append((study_uid, routes))

total = agree + disagree
print(f'series with >=2 independently-resolving routes: {total} (sampled first {sample_n} studies)')
if total:
    print(f'agree: {agree} ({agree / total:.1%}), disagree: {disagree} ({disagree / total:.1%})')
for example in disagreement_examples:
    print(example)


In [ ]:
import numpy as np

all_slice_counts = [c for counts in census_df['slice_counts'] for c in counts]
sc = np.array(all_slice_counts)
print(f'series-level slice counts, n={len(sc)}:')
for p in (0, 5, 25, 50, 75, 95, 100):
    print(f'  p{p}: {np.percentile(sc, p):.0f}')

per_study_total = census_df['slice_counts'].apply(sum)
print()
print('per-study total slice counts (all series summed):')
for p in (0, 5, 25, 50, 75, 95, 100):
    print(f'  p{p}: {np.percentile(per_study_total, p):.0f}')


In [ ]:
census_df.to_csv('/kaggle/working/laterality_census.csv', index=False)
series_route_df.to_csv('/kaggle/working/series_laterality_routes.csv', index=False)
print('saved laterality_census.csv and series_laterality_routes.csv to /kaggle/working -- download and copy into results/ locally')
